# S01 — synthetic scene contracts and first visualization

**Question:** Are a tiny road scene, viewpoints, causal records and unknown states reproducible and auditable? **Track:** GT-free contract only; evaluator geometry is isolated in `EvaluatorTruth`. **Gate:** G1 passes only after targeted tests and fresh-kernel execution with named exports. This stage makes no detector, conformity score, alarm, attack or quota claim. Units: m, m/s, integer ns; intensity is unitless.

In [ ]:
import json, os
from pathlib import Path
from IPython.display import Image, display, Markdown
from valid_point.provenance import export_table, sha256_file
from valid_point.synthetic import (load_synthetic_config, generate_scene, scene_contract_rows,
    unknown_state_rows, point_rows, scene_figure, availability_measurement, kinematics_availability)
from valid_point.io.pointcloud import write_pointcloud, read_pointcloud
root = Path(os.environ['VP_REPO_ROOT'])
run_dir = Path(os.environ['VP_RUN_DIR'])
config_path = root / 'configs/synthetic.json'
config = load_synthetic_config(config_path)
seed = 0
scene = generate_scene(config, seed)
provenance = {'run_id': run_dir.name, 'stage': 'S01', 'track': 'gt_free_contract_only',
    'segment': config['episode'] + '_0', 'config_sha256': sha256_file(config_path),
    'source_sha256': os.environ['VP_SOURCE_SHA256'], 'notebook_section': 'NB01'}
print('Run:', run_dir.name, 'seed:', seed, 'split:', scene.split_role)
print('Exact config and all seed families:')
print(json.dumps(config, indent=2, sort_keys=True))
print('Reference and calibration:', config['reference_lineage'], '/', config['calibration_lineage'])


## T01_scene_contract — causal inputs and coordinate frames

The decision unit is episode/sender/frame/deadline. Exact receiver availability at the deadline is allowed. Every sender has a distinct security group in this toy setup; distinct installations are a declared assumption, not inferred from point counts. Native axes are right-handed x-forward/y-left/z-up. `T_receiver_from_sensor` maps native metres into receiver metres; the receiver frame is road-aligned. The fixture vehicle has no verified transform or causal kinematics. The complete table shows all message states, clocks and unknown visibility.

In [ ]:
for transform in scene.transforms:
    print(transform.transform_id, transform.source_frame, '→', transform.target_frame,
          'valid ns:', transform.valid_from_ns, transform.valid_until_ns,
          'units/handedness/quaternion:', transform.units, transform.handedness, transform.quaternion_order,
          'provenance:', transform.provenance)
    print(*transform.matrix, sep='\n')
contract_rows = scene_contract_rows(scene)
for row in contract_rows: print(row)
contract_paths = export_table(contract_rows, 'T01_scene_contract', root / 'reports/tables', provenance)
print('Exports:', *contract_paths, sep='\n')


## F01_scene_views and point table

The shaded road is z=0 m, wall x=0 m spans y=-2…2 m, and the object moves +x between two displayed centers. The two west lines of sight cross the wall; north has the unique honest object view. These occlusion labels and moving-object centers are **evaluator truth** and never populate the decision input's visibility field. Each parsed cloud also contains road and static wall returns. Blue/purple arrows indicate local native +x/+y.

In [ ]:
point_table = point_rows(scene)
for row in point_table: print(row)
point_paths = export_table(point_table, 'T01_point_coordinates', root / 'reports/tables',
                           {**provenance, 'notebook_section': 'NB01_points'})
figure_paths = scene_figure(scene, root / 'reports/figures',
                            {**provenance, 'notebook_section': 'NB01_geometry'})
display(Image(filename=str(figure_paths[2])))
print('Point table:', *point_paths, sep='\n')
print('Figure:', *figure_paths, sep='\n')
print('Evaluator-only occluded IDs:', scene.evaluator_truth.occluded_sensor_ids,
      '; unique view:', scene.evaluator_truth.uniquely_useful_sensor_id)


## T01_unknown_states — raw measurements and counterexample

`present_empty` has a valid parsed count of **0 points** at the deadline. `absent` has **null count** and no arrival. The same numeric default would erase this distinction. Late and malformed payloads also have null decision-time counts. Kinematics for static sensors is structurally not applicable; missing vehicle samples are unknown. Zero returns do not establish visibility or dishonesty. Normalized evidence, conformity score, alarm/state and next-frame allocation are all **not run** in S01.

In [ ]:
state_rows = unknown_state_rows(scene)
for row in state_rows: print(row)
unknown_paths = export_table(state_rows, 'T01_unknown_states', root / 'reports/tables',
                             {**provenance, 'notebook_section': 'NB01_unknowns'})
empty, absent, late, malformed, no_kin = scene.observations[3:]
assert availability_measurement(empty).value == 0.0
assert availability_measurement(absent).value is None
assert all(row.visibility == 'unknown' for row in scene.observations)
print('Counterexample: empty=', availability_measurement(empty), '\nabsent=', availability_measurement(absent))
print('Vehicle kinematics=', kinematics_availability(no_kin))
print('Normalized evidence: not run; T: not run; alarm/state: not run; allocation: not run')
print('Exports:', *unknown_paths, sep='\n')


## Canonical reader round trip and hand checks

The JSON producer and reader preserve the complete point field order and count. These synthetic clouds are local source inputs for later overlays; no external LiDAR archive is involved. We check coordinate inverse round trips and the north object's eight extra returns. The table above contains the full 20 north points, not a truncated preview.

In [ ]:
from numpy import allclose
cloud_hashes = {}
for cloud_id, cloud in scene.clouds:
    path = run_dir / f'{cloud_id}.json'
    write_pointcloud(path, cloud)
    loaded = read_pointcloud(path)
    assert loaded == cloud and loaded.count == cloud.count
    cloud_hashes[cloud_id] = sha256_file(path)
for obs, transform in zip(scene.observations[:3], scene.transforms):
    xyz = scene.cloud(obs.cloud_id).xyz()
    assert allclose(transform.apply(transform.apply(xyz), inverse=True), xyz, atol=1e-12)
assert scene.observations[2].point_count == scene.observations[0].point_count + 8
print('Canonical cloud hashes/counts:', [(k, scene.cloud(k).count, h) for k,h in cloud_hashes.items()])
print('Coordinate inverse and eight-return visibility hand checks: PASS')


## Evidence-based G1 gate

The test report is generated before the notebook. The runner writes the final manifest and gate after fresh-kernel execution. A notebook error leaves a blocked run. This geometry is a deliberately simple generator; evaluator truth proves only the constructed occlusion, not GT-free visibility or a score finding.

In [ ]:
report = json.loads((run_dir / 'test_results.json').read_text())
print('Targeted test command:', ' '.join(report['command']))
print(report['output'])
assert report['exit_code'] == 0
assert len(contract_rows) == 8 and len(state_rows) == 8 and len(point_table) == 20
assert len(contract_paths) == 3 and len(unknown_paths) == 3 and len(figure_paths) == 4
print('NB01 visible G1 contract checks: PASS')
print('Limitations: visibility remains unknown online; no score/attack/quota, reference or calibration was run.')
